In [52]:
import json
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from tavily import TavilyClient
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()

llm = ChatOllama(model='mistral:latest',temperature=0.2)

TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
tavily_client = TavilyClient(api_key = TAVILY_API_KEY)

@tool
def calculator(expression:str)-> str:
  """Useful for executing mathematical expressions. Input should be a valid python math expression string."""
  return str(eval(expression))

@tool
def search_web(query: str)->str:
  """Useful when you need to answer questions about current events or when you lack specific knowledge."""
  try:
    web_response = tavily_client.search(query,
                                        search_depth="advanced")
    
    contents = []
    sources = []
    for result in web_response.get('results'):
      contents.append(result['content'])
      sources.append(result['url'])
    
    results = json.dumps({"contents":contents,"sources":sources})
    return results
    
  except errors.BadRequestError as e:
    print(f"Exception {e} Occured While Searching the Web")
    return f'Search Failed :{e}'

## a list of tools
tools = [search_web, calculator]

In [53]:
## system instruction prompt : for structured output
system_instruction = """You are an expert research assistant and note-taker.
  Your job is to read raw web content and distill it into 
  clean, structured study notes for a learner.
  
  Your ONLY output must be valid JSON — no prose, no markdown 
  fences, no explanation outside the JSON.
  
  Respond with this exact schema:
  {{
    "subtopic": string
    "summary": string,
    "key_concepts": [string],
    "important_facts": [string],
    "study_tips": [string],
    "sources_used": [string]
  }}
  
  
  Rules:
  - summary must be 3-5 sentences maximum
  - key_concepts must have 3-5 items
  - important_facts must have 3-5 items  
  - study_tips must have 2-3 items
  - Never invent facts not present in the provided content
  - If content is insufficient, say so in summary field"""

human_prompt = """Subtopic: {subtopic}

  Raw content:
  {tavily_results}

  Level: {level} 

  Then output ONLY the JSON notes — no reasoning in response."""  

In [54]:
subtopic = "Java-Programming-DSA"
level = 'Beginner'

tavily_results = search_web.invoke({"query":subtopic})

user_inputs = {
  "subtopic":subtopic, 
  "level": level, 
  "tavily_results": tavily_results
}

chat_prompt_template = ChatPromptTemplate.from_messages([
  ('system',system_instruction),
  ('human',human_prompt)
])

parser = JsonOutputParser()

chain = chat_prompt_template | llm | parser

In [57]:
response = chain.invoke(user_inputs)

In [58]:
for k,v in response.items():
  print(f"{k}:{v}")

subtopic:Java DSA Course Overview
summary:Coding Blocks offers DSA offline classes at Delhi and Noida centers for Java programming. The course covers DS in Java, recursion, searching, sorting, trees, graphs, greedy methods, dynamic programming, and interview preparation. It is recommended for beginners.
key_concepts:['Java Programming', 'Data Structures', 'Algorithms', 'Recursion', 'Searching', 'Sorting', 'Trees', 'Graphs', 'Greedy Methods', 'Dynamic Programming']
important_facts:['Coding Blocks offers classroom, live, and online options', 'The course prepares students for interviews with coding challenges, DSA interview questions, Java interview questions, and Spring Boot interview questions']
study_tips:['Practice coding challenges and interview questions', 'Review key concepts regularly', 'Attend live sessions or classroom training if possible']
sources_used:['https://www.codingblocks.com/data-structures-and-algorithms-using-java.html']


In [56]:
print(type(tavily_results))

<class 'str'>
